In [1]:
import numpy as np 
import pandas as pd 

# EXAMPLE, do not run 

In [14]:
# import webdriver
import time
import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.common.by import By
# import Action chains 
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

# import exeptions if element is not found
from selenium.common.exceptions import NoSuchElementException


###Add Tags here
taglist =["MSFT", "PYPL", "TSLA"]

IncomeStatementSummary = pd.DataFrame()
BalanceSheetSummary = pd.DataFrame()
IncomeStatementQuarterlySummary =pd.DataFrame()
BalanceSheetQuarterlySummary = pd.DataFrame()
StatisticsSummary = pd.DataFrame()

is_link ='https://finance.yahoo.com/lookup'
driver = webdriver.Edge()
driver.get(is_link)
time.sleep(5)



####For Germans only####
# Press Reject Button
RejectAll = driver.find_element(By.XPATH, '//button[@class="btn secondary reject-all"]')
# create action chain object
action = ActionChains(driver)
# click the item
action.click(on_element = RejectAll)
# perform the operation
action.perform()
time.sleep(5)
###########

#######Search for an elements of taglist to disable the button that appears after first search
def SearchElements(tag):
    SearchBar = driver.find_element(By.ID, "yfin-usr-qry")
    SearchBar.send_keys(tag)
    SearchBar.send_keys(Keys.ENTER)
    time.sleep(5)

    if tag == taglist[0]:
        MayBeLaterBtn = driver.find_element(By.XPATH, '//button[@class="Mx(a) Fz(16px) Fw(600) Mt(20px) D(n)--mobp"]')
        action = ActionChains(driver)
        action.click(on_element = MayBeLaterBtn)
        action.perform()
        time.sleep(5)

#####Get Statistics
def getStatistics(tag):
        global StatisticsSummary
        Statistics = pd.DataFrame(pd.np.empty((0, 61)))
        StatisticsBtn = driver.find_element(By.XPATH,  '//ul[@class="List(n) Whs(nw) fin-tab-items W(100%) Lh(1.7) H(44px) Bdbs(s) BdB(4px) Cf Mb(15px) Bdbc($seperatorColor) "]/li[4]/a[1]')
        action = ActionChains(driver)
        action.click(on_element = StatisticsBtn)
        action.perform()
        time.sleep(5)

        dataStatistics = driver.find_elements(By.XPATH, '//td[contains(@class, "Pos(st) Start(0) Bgc($lv2BgColor) fi-row:h_Bgc($hoverBgColor) Pend(10px)  Miw(140px)") or contains(@class, "Fw(500) Ta(end) Pstart(10px) Miw(60px)") or contains(@class, "Pos(st) Start(0) Bgc($lv2BgColor) fi-row:h_Bgc($hoverBgColor) Pend(10px) ")]')
        DataStatisticsList =[]

        #Collect all Names and Values from Statistics Site
        for value in dataStatistics:
         DataStatisticsList.append(value.text)
        
        #Create Columns Names and Append tag to it
        ColumnNames = DataStatisticsList[::2]
        ColumnNames.insert(0, "tag")

        #Create a List that contains all Statistics Values
        Data = DataStatisticsList[1::2]
        Data.insert(0,tag)

        #Add Columns and Data
        Statistics.columns = ColumnNames 
        Statistics.loc[len(Statistics)] = Data
       # StatisticsSummary = StatisticsSummary.append(Statistics, ignore_index=True)
        StatisticsSummary = pd.concat([StatisticsSummary,Statistics], ignore_index=True, sort=False)

#####Get IncomeStatement Yearly
def getIncomeStatementYearly(tag):
      
        #Go to Financials
        FinancialsButton = driver.find_element(By.XPATH, '//ul[@class="List(n) Whs(nw) fin-tab-items W(100%) Lh(1.7) H(44px) Bdbs(s) BdB(4px) Cf Mb(15px) Bdbc($seperatorColor) "]/li[7]/a[1]')
        action = ActionChains(driver)
        action.click(on_element = FinancialsButton)
        action.perform()
        time.sleep(3)

        global IncomeStatementSummary

        #Get Income Common Stockholders 
        IncomeCommonStockholders = driver.find_element(By.XPATH, '//button[@aria-label="Net Income Common Stockholders"]')
        action = ActionChains(driver)
        action.click(on_element = IncomeCommonStockholders)
        action.perform()
        time.sleep(0.2)


        ####Here we collect the income statement in full########
        Data = driver.find_elements(By.XPATH, '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg Bgc($lv1BgColor) fi-row:h_Bgc($hoverBgColor) D(tbc)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(tbc)")]')
        Column_Headers = driver.find_elements(By.XPATH, '//div[contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(215px)--mv2 W(200px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(200px)--mv2 W(185px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(185px)--mv2 W(170px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(170px)--mv2 W(155px) undefined")]')
        Dates = driver.find_elements(By.XPATH,  '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b) Tt(u) Bgc($lv1BgColor)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b)")]')

        IncomeStatementHeaders = []
        IncomeStatementData =[]
        DatesColumn=[]

        for value in Column_Headers:
             IncomeStatementHeaders.append(value.text)

        for value in Data:
             IncomeStatementData.append(value.text)
     
        for value in Dates:
             DatesColumn.append(value.text)
     
        ######Chunck size is len of Dates column
        chunk_size = len(DatesColumn)
        ListOfLists = list()

        for i in range(0, len(IncomeStatementData), chunk_size):
            ListOfLists.append(IncomeStatementData[i:i+chunk_size])
 
        IncomeStatementYearly = pd.DataFrame()
 
        #Add Columns with Chunk size
        for z in range(0, len(IncomeStatementHeaders), 1):
            IncomeStatementYearly[z] = ListOfLists[z]


        IncomeStatementYearly.columns = IncomeStatementHeaders
        IncomeStatementYearly.insert(0, "tag", tag)
        IncomeStatementYearly.insert(1, "Dates", DatesColumn)

        IncomeStatementSummary = pd.concat([IncomeStatementSummary,IncomeStatementYearly], axis=0, ignore_index=True, sort=False)
#####Get IncomeStatement Quarterly
def getIncomeStatementQuarterly(tag):
      
        #Go to Quarterly
        QuarterlyButton = driver.find_element(By.XPATH, '//button[@class="P(0px) M(0px) C($linkColor) Bd(0px) O(n)"]')
        action = ActionChains(driver)
        action.click(on_element = QuarterlyButton)
        action.perform()
        time.sleep(5)

        global IncomeStatementQuarterlySummary

        ####Here we collect the income statement in full########
        Data = driver.find_elements(By.XPATH, '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg Bgc($lv1BgColor) fi-row:h_Bgc($hoverBgColor) D(tbc)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(tbc)")]')
        Column_Headers = driver.find_elements(By.XPATH, '//div[contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(215px)--mv2 W(200px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(200px)--mv2 W(185px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(185px)--mv2 W(170px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(170px)--mv2 W(155px) undefined")]')
        Dates = driver.find_elements(By.XPATH,  '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b) Tt(u) Bgc($lv1BgColor)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b)")]')

        IncomeStatementHeaders = []
        IncomeStatementData =[]
        DatesColumn=[]

        for value in Column_Headers:
             IncomeStatementHeaders.append(value.text)

        for value in Data:
             IncomeStatementData.append(value.text)
     
        for value in Dates:
             DatesColumn.append(value.text)
     
        ######Chunck size is len of Dates column
        chunk_size = len(DatesColumn)
        ListOfLists = list()

        for i in range(0, len(IncomeStatementData), chunk_size):
            ListOfLists.append(IncomeStatementData[i:i+chunk_size])
 
        IncomeStatementQuarterly = pd.DataFrame()
 
        #Add Columns with Chunk size
        for z in range(0, len(IncomeStatementHeaders), 1):
            IncomeStatementQuarterly[z] = ListOfLists[z]


        IncomeStatementQuarterly.columns = IncomeStatementHeaders
        IncomeStatementQuarterly.insert(0, "tag", tag)
        IncomeStatementQuarterly.insert(1, "Dates", DatesColumn)

        IncomeStatementQuarterlySummary = pd.concat([IncomeStatementQuarterlySummary,IncomeStatementQuarterly], axis=0, ignore_index=True, sort=False)

def getBalanceSheet(tag):

      global BalanceSheetSummary
      #Go to Balance Sheet
      BalanceSheetBtn = driver.find_element(By.XPATH, '//div[@class="Fw(500) D(ib) Pend(10px) H(18px) BdEnd Bdc($seperatorColor)"]')
      action = ActionChains(driver)
      action.click(on_element = BalanceSheetBtn)
      action.perform()
      time.sleep(5)

      #Go to Balance Sheet
   
      #Open subaccounts of BalanceSheet
      try:
            TotalAssetsBtn = driver.find_element(By.XPATH,  '//button[@aria-label="Total Assets"]')
            action = ActionChains(driver)
            action.click(on_element = TotalAssetsBtn)
            action.perform()
            time.sleep(0.2)

            TotalLiabilitiesBtn = driver.find_element(By.XPATH,  '//button[@aria-label="Total Liabilities Net Minority Interest"]')
            action = ActionChains(driver)
            action.click(on_element = TotalLiabilitiesBtn)
            action.perform()
            time.sleep(0.2)

            TotalEquityBtn = driver.find_element(By.XPATH,  '//button[@aria-label="Total Equity Gross Minority Interest"]')
            action = ActionChains(driver)
            action.click(on_element = TotalEquityBtn)
            action.perform()
            time.sleep(0.2)

      except NoSuchElementException:
            print("Element not found")

      try:      
        Data = driver.find_elements(By.XPATH, '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(tbc)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg Bgc($lv1BgColor) fi-row:h_Bgc($hoverBgColor) D(tbc)")]')
        Dates = driver.find_elements(By.XPATH,'//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b) Bgc($lv1BgColor)")]' )
        Column_Headers = driver.find_elements(By.XPATH,'//div[contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(215px)--mv2 W(200px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(200px)--mv2 W(185px) undefined")]' )

        ColumnHeadersList =[]
        DatesList = []
        BalanceSheetList = []

        for value in  Column_Headers:
         ColumnHeadersList.append(value.text)

        for value in Data:
         BalanceSheetList.append(value.text)

        for value in Dates:
         DatesList.append(value.text) 

    
        ListOfLists= list()
        chunk_size = len(DatesList)

        for i in range(0, len(BalanceSheetList), chunk_size):
            ListOfLists.append(BalanceSheetList[i:i+chunk_size])

        BalanceSheetYearly= pd.DataFrame()

        for z in range(0, len(ColumnHeadersList), 1):
         BalanceSheetYearly[z] = ListOfLists[z]
    
        BalanceSheetYearly.columns = ColumnHeadersList

        BalanceSheetYearly.insert(0, "tag", tag)
        BalanceSheetYearly.insert(1, "Dates", DatesList)

        BalanceSheetSummary = pd.concat([BalanceSheetSummary,BalanceSheetYearly], axis=0, ignore_index=True, sort=False)

      except NoSuchElementException:
        print("Element not found")

def getBalanceSheetQuarterly(tag):

      global BalanceSheetQuarterlySummary
      #Go to Quarterly
      QuarterlyButton = driver.find_element(By.XPATH, '//button[@class="P(0px) M(0px) C($linkColor) Bd(0px) O(n)"]')
      action = ActionChains(driver)
      action.click(on_element = QuarterlyButton)
      action.perform()
      time.sleep(2)

      try:
        Data = driver.find_elements(By.XPATH, '//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(tbc)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg Bgc($lv1BgColor) fi-row:h_Bgc($hoverBgColor) D(tbc)")]')
        Dates = driver.find_elements(By.XPATH,'//div[contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b)") or contains(@class, "Ta(c) Py(6px) Bxz(bb) BdB Bdc($seperatorColor) Miw(120px) Miw(100px)--pnclg D(ib) Fw(b) Bgc($lv1BgColor)")]' )
        Column_Headers = driver.find_elements(By.XPATH,'//div[contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(215px)--mv2 W(200px) undefined") or contains(@class, "D(ib) Va(m) Ell Mt(-3px) W(200px)--mv2 W(185px) undefined")]' )

        ColumnHeadersList =[]
        DatesList = []
        BalanceSheetList = []

        for value in  Column_Headers:
         ColumnHeadersList.append(value.text)

        for value in Data:
         BalanceSheetList.append(value.text)

        for value in Dates:
         DatesList.append(value.text) 
         
        ListOfLists= list()
        chunk_size = len(DatesList)

        for i in range(0, len(BalanceSheetList), chunk_size):
            ListOfLists.append(BalanceSheetList[i:i+chunk_size])

        BalanceSheetQuarterly= pd.DataFrame()

        for z in range(0, len(ColumnHeadersList), 1):
         BalanceSheetQuarterly[z] = ListOfLists[z]
    
        BalanceSheetQuarterly.columns = ColumnHeadersList

        BalanceSheetQuarterly.insert(0, "tag", tag)
        BalanceSheetQuarterly.insert(1, "Dates", DatesList)

        BalanceSheetQuarterlySummary = pd.concat([BalanceSheetQuarterlySummary,BalanceSheetQuarterly], axis=0, ignore_index=True, sort=False)

      except NoSuchElementException:
        print("Element not found")

for tag in taglist:
    SearchElements(tag)
    getStatistics(tag)
    getIncomeStatementYearly(tag)
    getIncomeStatementQuarterly(tag)
    getBalanceSheet(tag)
    getBalanceSheetQuarterly(tag)


StatisticsSummary.to_excel("C:/Users/User/Desktop/Python/WebScraping/Stats.xlsx",
                 sheet_name='Sheet_name_1')  

IncomeStatementSummary.to_excel("C:/Users/User/Desktop/Python/WebScraping/IncomeStatement.xlsx",
                 sheet_name='Sheet_name_1')  

IncomeStatementQuarterlySummary.to_excel("C:/Users/User/Desktop/Python/WebScraping/IncomeStatementQuarterly.xlsx",
                 sheet_name='Sheet_name_1')  

BalanceSheetSummary.to_excel("C:/Users/User/Desktop/Python/WebScraping/BalanceSheet.xlsx",
                 sheet_name='Sheet_name_1')  

BalanceSheetQuarterlySummary.to_excel("C:/Users/User/Desktop/Python/WebScraping/BalanceSheetQuarterly.xlsx",
                 sheet_name='Sheet_name_1')  

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="yfin-usr-qry"]"}
  (Session info: MicrosoftEdge=131.0.2903.70); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	(No symbol) [0x00007FF786716B15]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF786A3F4A4+1437348]
	sqlite3_dbdata_init [0x00007FF786AE2DE6+643190]
	(No symbol) [0x00007FF78663C9DB]
	(No symbol) [0x00007FF78663CAE3]
	(No symbol) [0x00007FF7866792F7]
	(No symbol) [0x00007FF78665C1DF]
	(No symbol) [0x00007FF786633437]
	(No symbol) [0x00007FF786676BFF]
	(No symbol) [0x00007FF78665BE03]
	(No symbol) [0x00007FF786632984]
	(No symbol) [0x00007FF786631E30]
	(No symbol) [0x00007FF786632571]
	Microsoft::Applications::Events::EventProperty::empty [0x00007FF7869EBB34+1094964]
	(No symbol) [0x00007FF7867532C8]
	Microsoft::Applications::Events::Ev

# yfinance

In [25]:
import yfinance as yf

ticker_symbol = 'MSFT'
ticker = yf.Ticker(ticker_symbol)
historical_data = ticker.history(period='5y')
flat_data = historical_data.reset_index()
flat_data.insert(0, 'stock', ticker_symbol)

In [26]:
flat_data

,stock,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,MSFT,2019-12-02 00:00:00-05:00,145.204056,145.223190,141.865931,143.042404,27418400,0.0,0.0
1,MSFT,2019-12-03 00:00:00-05:00,141.072025,142.927594,140.268566,142.812820,24066000,0.0,0.0
2,MSFT,2019-12-04 00:00:00-05:00,143.606720,143.644973,142.707621,143.329346,17574700,0.0,0.0
3,MSFT,2019-12-05 00:00:00-05:00,143.520626,143.778881,142.975423,143.405838,17869100,0.0,0.0
4,MSFT,2019-12-06 00:00:00-05:00,144.419744,145.261441,143.731073,145.146667,16403500,0.0,0.0
...,...,...,...,...,...,...,...,...,...
1253,MSFT,2024-11-22 00:00:00-05:00,411.369995,417.399994,411.059998,417.000000,24814600,0.0,0.0
1254,MSFT,2024-11-25 00:00:00-05:00,418.380005,421.079987,414.850006,418.790009,27691100,0.0,0.0
1255,MSFT,2024-11-26 00:00:00-05:00,419.589996,429.040009,418.850006,427.989990,23458900,0.0,0.0
1256,MSFT,2024-11-27 00:00:00-05:00,425.109985,427.230011,422.019989,422.989990,18332400,0.0,0.0


In [28]:
flat_data.to_csv('stock_values.csv', index=False)

# Working part

In [18]:
import time
import random
import pandas as pd

# import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# import Action chains 
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException

# create webdriver object
driver = webdriver.Chrome()

# get yahoo finance.com
is_link ='https://finance.yahoo.com/lookup'
driver.get(is_link)
time.sleep(5)

# get element 
RejectAll= driver.find_element(By.XPATH, '//button[@class="btn secondary reject-all"]')
# create action chain object
action = ActionChains(driver)
# click the item
action.click(on_element = RejectAll)
# perform the operation
action.perform()
time.sleep(5)
#################################
tag ='MSFT'

SearchBar = driver.find_element(By.ID, "ybar-sbq")
SearchBar.send_keys(tag)
SearchBar.send_keys(Keys.ENTER)
# time.sleep(5)


# Get to Historical Data 

aside = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//aside[@class="left yf-cfn520 stickyNavbar"]'))
)

nav_bar = driver.find_element(By.XPATH, '//section[@data-testid="quote-nav-bar"]/nav')

# Find all list items (li) within the navigation bar
nav_items = nav_bar.find_elements(By.TAG_NAME, 'li')

# Iterate through the list items to find the one with title 'Historical Data'
for item in nav_items:
    title = item.find_element(By.TAG_NAME, 'a').get_attribute('title')
    if title == "Historical Data":
        item.click()  # Click the Historical Data tab
        break   

# Handle Historical Data 

# Wait until the menuContainer div is present on the page
menu_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "menuContainer") and contains(@class, "yf-9a5vow")]'))
)

# Locate the button within the menuContainer and click it
menu_button = menu_container.find_element(By.XPATH, './/button[contains(@class, "menuBtn")]')
menu_button.click()

"""# Wait for the Sing in popup to appear
popup = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "dialog-container"))
)

# Locate an area outside the popup (e.g., the price container)
outside_area = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "price"))
)

# Create an ActionChains object
action = ActionChains(driver)

# Click outside the popup
action.move_to_element(outside_area).click().perform()"""

"""def close_all_popups(max_attempts=5):
    attempts = 0  # Counter to track how many times we've handled popups
    while attempts < max_attempts:
        try:
            # Wait for the popup to appear
            popup = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CLASS_NAME, "dialog-container"))
            )
            # Locate an area outside the popup
            outside_area = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "price"))
            )
            # Click outside the popup to close it
            action = ActionChains(driver)
            action.move_to_element(outside_area).click().perform()
            print(f"Popup closed. Attempt {attempts + 1} of {max_attempts}.")
            attempts += 1  # Increment attempt counter
        except TimeoutException:
            # If no popup is found, break the loop
            print("No popup found, moving on.")
            break

# Call the function to handle all popups
close_all_popups()"""


# Wait for the popup to appear
popup = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "dialog-container"))
)
popup_html = popup.get_attribute('outerHTML')

# time.sleep(random.uniform(1, 2))

# Locate an area outside the popup
outside_area = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "price"))
)
# Simulate hover
action = ActionChains(driver).move_to_element(outside_area).click().perform()

# Send ESCAPE key to dismiss the popup
body = driver.find_element(By.TAG_NAME, 'body')
body.send_keys(Keys.ESCAPE)

# Date range 
# Wait until the menuContainer div is present on the page
menu_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "menuContainer") and contains(@class, "yf-9a5vow")]'))
)

# Locate the button within the menuContainer and click it
menu_button = menu_container.find_element(By.XPATH, './/button[contains(@class, "menuBtn")]')
menu_button.click()



time.sleep(5)



In [ ]:
# Sign in 
# Wait for the dialog container to become visible
dialog = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[@class="dialog-container yf-9a5vow" and @aria-hidden="false"]'))
)

# Locate the 'Sign in' link inside the dialog container
sign_in_link = WebDriverWait(dialog, 10).until(
    EC.element_to_be_clickable((By.XPATH, './/a[contains(@class, "secondary-btn-link") and @data-v9y="1" and contains(., "Sign in")]'))
)

# Click the 'Sign in' link
sign_in_link.click()
time.sleep(5)


#  Sign in 
yahoo_login = 'kacperkubicki'
yahoo_password = 'Kacperson1998!'

# Wait for the input field to be present and visible
username_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "login-username"))
)

# Enter the variable 'kacper' into the input field
username_field.send_keys(yahoo_login)

# Wait for the Next button to be clickable and then click it
next_button = driver.find_element(By.ID, 'login-signin')
next_button.click()

# Wait for the input field to be present and visible
password_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "login-passwd"))
)

# Enter the variable 'kacper' into the input field
password_field.send_keys(yahoo_password)

# Wait for the Next button to be clickable and then click it
next_button = driver.find_element(By.ID, 'login-signin')
next_button.click()

# Wait until the menuContainer div is present on the page
menu_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "menuContainer") and contains(@class, "yf-9a5vow")]'))
)

# Locate the button within the menuContainer and click it
menu_button = menu_container.find_element(By.XPATH, './/button[contains(@class, "menuBtn")]')
menu_button.click()

"""# random pop up window, just need to click elsewhere
container = driver.find_element(By.CSS_SELECTOR, '.container.yf-1tejb6')
# Click on the first <fin-streamer> element inside the container
clickable_element = container.find_element(By.TAG_NAME, 'fin-streamer')
clickable_element.click()

# Wait until the menuContainer div is present on the page
menu_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "menuContainer") and contains(@class, "yf-9a5vow")]'))
)

# Locate the button within the menuContainer and click it
menu_button = menu_container.find_element(By.XPATH, './/button[contains(@class, "menuBtn")]')
menu_button.click()"""

time.sleep(5)


In [ ]:

# Wait until the menuContainer div is present on the page
menu_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "menuContainer") and contains(@class, "yf-9a5vow")]'))
)

# Locate the button within the menuContainer and click it
menu_button = menu_container.find_element(By.XPATH, './/button[contains(@class, "menuBtn")]')
menu_button.click()

time.sleep(5)





In [ ]:


# Label and Value on Summary Tab 

# Locate the parent div using XPath
quote_statistics = driver.find_element(By.XPATH, '//div[@data-testid="quote-statistics"]/ul')

# Find all <li> elements within this container
items = quote_statistics.find_elements(By.XPATH, './li')

data = []
for item in items:
    # Extract label and value
    label = item.find_element(By.CLASS_NAME, 'label').text
    value = item.find_element(By.CLASS_NAME, 'value').text
    data.append({'Label': label, 'Value': value})

# Convert list of dictionaries to a DataFrame
df = pd.DataFrame(data)

# Print or save the DataFrame
print(df)

# Close the browser
driver.quit()


![example-MSFT-html-structure](example-MSFT-html-structure.png) 

In [ ]:
import time

# import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
# import Action chains 
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

# create webdriver object
driver = webdriver.Edge()

# get yahoo finance.com
is_link ='https://finance.yahoo.com/lookup'
driver.get(is_link)
time.sleep(5)

# get element 
RejectAll= driver.find_element(By.XPATH, '//button[@class="btn secondary reject-all"]')
# create action chain object
action = ActionChains(driver)
# click the item
action.click(on_element = RejectAll)
# perform the operation
action.perform()
time.sleep(5)
#################################
tag ='MSFT'

SearchBar = driver.find_element(By.ID, "ybar-sbq")
SearchBar.send_keys(tag)
SearchBar.send_keys(Keys.ENTER)
time.sleep(5)


# 

# Locate the parent div using XPath
quote_statistics = driver.find_element(By.XPATH, '//div[@data-testid="quote-statistics"]/ul')

# Find all <li> elements within this container
items = quote_statistics.find_elements(By.XPATH, './li')

data = []
for item in items:
    # Extract label and value
    label = item.find_element(By.CLASS_NAME, 'label').text
    value = item.find_element(By.CLASS_NAME, 'value').text
    data.append({'Label': label, 'Value': value})

# Convert list of dictionaries to a DataFrame
df = pd.DataFrame(data)

# Print or save the DataFrame
print(df)

# Close the browser
driver.quit()


In [ ]:

# MayBeLaterBtn = driver.find_element(By.XPATH, '//button[@class="Mx(a) Fz(16px) Fw(600) Mt(20px) D(n)--mobp"]')
# action = ActionChains(driver)
# action.click(on_element = MayBeLaterBtn)
# action.perform()
# time.sleep(5)


# Table = driver.find_elements(By.XPATH, '//td[contains(@class, "C($primaryColor) W(51%)") or contains(@class, "Ta(end) Fw(600) Lh(14px)")]')
Table = driver.find_elements(By.XPATH, '//ul[contains(@class, "C($primaryColor) W(51%)") or contains(@class, "Ta(end) Fw(600) Lh(14px)")]')
# Table = driver.find_elements(By.XPATH, '//td[contains(@class, "yf-11uk5vd")]')
TableList =[]

#Collect all Names and Values
for value in Table:
    TableList.append(value.text)
    print (value.text)

time.sleep(100)

In [ ]:
quote_statistics = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="quote-statistics"] ul')

# Find all <li> elements within this container
items = quote_statistics.find_elements(By.TAG_NAME, 'li')

# Iterate through each item to extract the label and value
data = {}
for item in items:
    # Extract label and value
    label = item.find_element(By.CLASS_NAME, 'label').text
    value = item.find_element(By.CLASS_NAME, 'value').text
    data[label] = value

# Print or process the scraped data as needed
print(data)

# Close the browser
driver.quit()